# Compare the baseline across E01, E02, and E03

Provide existing run IDs for each environment. Set `PILOT_LOCAL_RUN_IDS`,
`PILOT_DOCKER_RUN_IDS`, and `PILOT_KUBERNETES_RUN_IDS` to comma-separated
IDs in round order. Use the singular `_RUN_ID` variables for a single run.
This notebook reads existing DAG runs and task states from the Airflow APIs.

[English guide](../../docs/operations-en.md) · [한국어 가이드](../../docs/operations-ko.md)

In [ ]:
import os

import pandas as pd
from IPython.display import display

from pilot_kit.airflow_api import DAG_ID, collect, connect, get_run, request, run_path

environments = {"local": "E01", "docker": "E02", "kubernetes": "E03"}
run_ids = {}
for target in environments:
    prefix = f"PILOT_{target.upper()}_RUN_ID"
    values = os.environ.get(f"{prefix}S") or os.environ.get(prefix, "")
    ids = tuple(value.strip() for value in values.split(",") if value.strip())
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate run IDs for {environments[target]}.")
    run_ids[target] = ids

if not any(run_ids.values()):
    raise ValueError("Enter at least one existing DAG run ID.")

In [ ]:
expected_tasks = {"partition", "sum_squares", "total"}
rows = []
for target, ids in run_ids.items():
    if not ids:
        continue
    with connect(target) as client:
        for repeat, run_id in enumerate(ids, start=1):
            result = collect(client, target, get_run(client, run_id))
            instances = request(client, "GET", f"{run_path(DAG_ID, run_id)}/taskInstances")
            task_states = [(task["task_id"], task["state"]) for task in instances["task_instances"]]
            successful_tasks = {task for task, state in task_states if state == "success"}
            rows.append(
                {
                    **result,
                    "round": repeat,
                    "task_states": ", ".join(
                        f"{task}: {state}" for task, state in sorted(task_states)
                    ),
                    "tasks_verified": len(task_states) == len(expected_tasks)
                    and successful_tasks == expected_tasks,
                }
            )

runs = pd.DataFrame(rows).set_index("environment").rename(index=environments)
with pd.option_context("display.max_colwidth", None):
    display(runs)

In [ ]:
expected = "333283335000"
verified = (
    runs["state"].eq("success")
    & runs["items"].eq(10000)
    & runs["shards"].eq(1)
    & runs["total"].eq(expected)
    & runs["tasks_verified"]
    & runs["duration_seconds"].notna()
)
counts = {len(ids) for ids in run_ids.values()}
balanced = len(counts) == 1 and 0 not in counts
if not verified.all():
    verdict = "Check run and task states, inputs, results, and durations"
else:
    verdict = "Passed" if balanced else "Incomplete"
print("Portability verdict:", verdict)

durations = runs.loc[verified, ["round", "duration_seconds"]]
caption = (
    "Infrastructure resources differ. E01 and E02 share a local Docker host.\n"
    "E03 uses separate Kubernetes nodes. Durations reflect both infrastructure and task execution."
)
if not durations.empty:
    if max(counts) > 1:
        stats = durations.groupby(level="environment", sort=False)["duration_seconds"].agg(
            ["count", "median", "mean", "min", "max"]
        )
        display(stats.round(3))
        points = durations.reset_index().pivot(
            index="round", columns="environment", values="duration_seconds"
        )
        ax = points.plot(
            style="-o",
            figsize=(8, 4.6),
            xlabel="Round",
            ylabel="DAG duration (s)",
            xticks=range(1, max(counts) + 1),
            title="Baseline DAG duration by round",
        )
    else:
        ax = durations["duration_seconds"].plot.bar(
            figsize=(8, 4.6), ylabel="DAG duration (s)", xlabel="Environment", rot=0
        )
    ax.set_ylim(bottom=0)
    figure = ax.get_figure()
    figure.subplots_adjust(bottom=0.23)
    figure.text(0.02, 0.04, caption, fontsize=9)